# Diffusion Kurtosis Imaging Evaluates the Neuroprotective Effects of Low-Intensity Treadmill Exercise in a 6-OHDA-Induced Rat Model of Parkinson’s Disease Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading, inspecting, and analyzing a Croissant-structured dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All references to dataset entities (record sets, fields, columns) use their canonical `@id` to ensure clarity and consistency.

### Dataset Source
This dataset is described by a Croissant schema accessible here:

https://sen.science/doi/10.71728/senscience.2c46-m4rp/fair2.json


In [ ]:
# Install the mlcroissant library if not already installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load the dataset and metadata using `mlcroissant`. This will fetch the Croissant schema and prepare the dataset interface for exploration and extraction.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.2c46-m4rp/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished} | Version: {metadata.version}\n")
print(f"Identifier: {metadata.identifier}")

## 2. Data Overview
Let's review available record sets, their fields, and the corresponding `@id` values. All references to entities will use their `@id` as defined in the Croissant schema.

In [ ]:
# List all available record sets and their field @ids
record_sets = list(dataset.record_sets)

print("Available Record Sets:")
for rs in record_sets:
    print(f"- Record Set name: {rs.name}, @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id})")
    print('')
if not record_sets:
    print("No record sets found in the dataset schema.")

## 3. Data Extraction
Load tabular data from each record set into a pandas DataFrame for analysis. We'll keep all references via their `@id` values.

In [ ]:
# Prepare to load records from each record set using @id
dataframes = dict()
for rs in dataset.record_sets:
    records = list(dataset.records(record_set=rs.id))
    dataframes[rs.id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records for Record Set: {rs.name} (@id: {rs.id})")
    if len(records) > 0:
        print(f"Fields: {list(dataframes[rs.id].columns)}\n")

# Select the first non-empty record set for demonstration
selected_rs = None
for rs_id, df in dataframes.items():
    if not df.empty:
        selected_rs = rs_id
        break
if selected_rs:
    print(f"Showing head of first non-empty record set: {selected_rs}")
    display(dataframes[selected_rs].head())
else:
    print("No non-empty record sets found in the data.")

## 4. Exploratory Data Analysis (EDA)
We'll perform basic EDA on a sample numeric field in a record set. Typical steps include filtering, normalization, and grouping. All operations reference fields via their `@id` as discovered in the overview step.

In [ ]:
# --- EDA Preparation ---
# Use the first non-empty record set and select a sample numeric field
if selected_rs:
    df = dataframes[selected_rs]
    # Identify numeric fields by dtype
    numeric_columns = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
    if not numeric_columns:
        print("No numeric fields found in this record set.")
    else:
        # Choose a numeric field to analyze
        numeric_field_id = numeric_columns[0]  # e.g., '@id' of a numeric column
        print(f"Analyzing numeric field: {numeric_field_id}")

        # Filter the dataset on a threshold (example: > 0.0)
        threshold = 0.0
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize the numeric field (z-score)
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Optionally group by a categorical field (if exists)
        # We'll take the first non-numeric, non-index column as group
        group_candidates = [col for col in df.columns if col != numeric_field_id and not np.issubdtype(df[col].dtype, np.number)]
        if group_candidates:
            group_field_id = group_candidates[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
            display(grouped_df)
        else:
            print("No group-able categorical fields found for grouping.")
else:
    print("No data to analyze.")

## 5. Visualization
Visualize data distributions or the relationship between fields, e.g., histogram or group means. All variables are referenced using their `@id`.

In [ ]:
# Simple visualizations using matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

if selected_rs and not filtered_df.empty:
    # Histogram of the selected numeric field
    plt.figure(figsize=(6,4))
    sns.histplot(filtered_df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouped_df created, show bar plot of group means
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()
else:
    print("Insufficient data for visualization.")

## 6. Conclusion

In this notebook, we loaded and explored a Croissant-described FAIR dataset on neuroimaging and behavioral assessment in a rat model of Parkinson's Disease. Using `mlcroissant`, we

- Loaded and summarized dataset metadata
- Enumerated available record sets and fields (by `@id`)
- Loaded tabular data to pandas DataFrame(s)
- Conducted preliminary EDA by filtering and normalizing a numeric field
- Visualized distributions and group-wise means where available

All data entity references (record sets, fields) used their original `@id` for full reproducibility. Explore further by applying your domain-specific analyses on the loaded DataFrames!